In [1]:
!pip install yfinance prophet xgboost tensorflow seaborn statsmodels joblib --quiet

In [2]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# ML Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Time Series
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import tensorflow as tf

warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [3]:
def calculate_metrics(y_true, y_pred):
    try:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        with np.errstate(divide='ignore', invalid='ignore'):
            mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
            if np.isnan(mape) or np.isinf(mape):
                mape = 0
        accuracy = max(0, 100 - mape)
        return {"RMSE": round(float(rmse), 2), "Accuracy": round(float(accuracy), 2)}
    except:
        return {"RMSE": 0.0, "Accuracy": 0.0}

In [4]:
# ---------------- Linear Regression ----------------
def predict_linear_regression(df, days_forecast=7):
    df = df.copy().sort_values('date')
    df['diff'] = df['close'].diff()

    for i in range(1, 4):
        df[f'lag_{i}'] = df['diff'].shift(i)

    df.dropna(inplace=True)
    if len(df) < 30: return None, None

    features = [f'lag_{i}' for i in range(1, 4)]
    target = 'diff'
    test_days = 14

    train_df = df.iloc[:-test_days]
    test_df = df.iloc[-test_days:]

    # Model
    model = LinearRegression()
    model.fit(train_df[features], train_df[target])

    # Recursive validation
    curr_diffs = train_df.iloc[-1][features].values
    curr_price = train_df.iloc[-1]['close']
    preds = []

    for _ in range(test_days):
        diff = model.predict([curr_diffs])[0]
        curr_price += diff
        preds.append(curr_price)
        curr_diffs = np.array([diff, curr_diffs[0], curr_diffs[1]])

    metrics = calculate_metrics(test_df['close'], preds)

    # Future
    model.fit(df[features], df[target])
    curr_diffs = df.iloc[-1][features].values
    curr_price = df.iloc[-1]['close']
    future = []

    for _ in range(days_forecast):
        diff = model.predict([curr_diffs])[0]
        curr_price += diff
        future.append(curr_price)
        curr_diffs = np.array([diff, curr_diffs[0], curr_diffs[1]])

    return future, metrics


# ---------------- Random Forest / XGBoost ----------------
def predict_tree_model(df, model_type='rf', days_forecast=7):
    df = df.copy().sort_values('date')

    for i in range(1, 4):
        df[f'lag_{i}'] = df['close'].shift(i)

    df.dropna(inplace=True)
    if len(df) < 30: return None, None

    features = [f'lag_{i}' for i in range(1, 4)]
    test_days = 14

    train_df = df.iloc[:-test_days]
    test_df = df.iloc[-test_days:]

    model = (
        RandomForestRegressor(n_estimators=100, random_state=42)
        if model_type == 'rf'
        else XGBRegressor(n_estimators=100, objective='reg:squarederror', random_state=42)
    )

    model.fit(train_df[features], train_df['close'])

    curr = train_df.iloc[-1][features].values
    preds = []

    for _ in range(test_days):
        p = model.predict([curr])[0]
        preds.append(p)
        curr = np.array([p, curr[0], curr[1]])

    metrics = calculate_metrics(test_df['close'], preds)

    # Future
    model.fit(df[features], df['close'])
    curr = df.iloc[-1][features].values
    future = []

    for _ in range(days_forecast):
        p = model.predict([curr])[0]
        future.append(p)
        curr = np.array([p, curr[0], curr[1]])

    return future, metrics


# ---------------- ARIMA ----------------
def predict_arima(df, days_forecast=7):
    data = df['close'].values
    if len(data) < 30: return None, None

    try:
        test = 14
        model = ARIMA(data[:-test], order=(5, 1, 0)).fit()
        preds = model.forecast(test)
        metrics = calculate_metrics(data[-test:], preds)

        model_full = ARIMA(data, order=(5, 1, 0)).fit()
        future = model_full.forecast(days_forecast).tolist()

        return future, metrics
    except:
        return None, None


# ---------------- Prophet ----------------
def predict_prophet(df, days_forecast=7):
    p_df = df.rename(columns={'date': 'ds', 'close': 'y'})

    if len(p_df) < 30: return None, None

    try:
        test = 14
        m = Prophet(daily_seasonality=True)
        m.fit(p_df[:-test])

        fc = m.predict(m.make_future_dataframe(test))
        preds = fc.tail(test)['yhat'].values

        metrics = calculate_metrics(p_df['y'].values[-test:], preds)

        m2 = Prophet(daily_seasonality=True)
        m2.fit(p_df)
        future = m2.predict(m2.make_future_dataframe(days_forecast)).tail(days_forecast)['yhat'].tolist()

        return future, metrics
    except:
        return None, None


# ---------------- LSTM ----------------
def predict_lstm(df, days_forecast=7):
    data = df['close'].values.reshape(-1, 1)
    lookback = 60

    if len(data) < lookback + 20: return None, None

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(data)

    def seq(ds):
        x, y = [], []
        for i in range(lookback, len(ds)):
            x.append(ds[i - lookback:i])
            y.append(ds[i])
        return np.array(x), np.array(y)

    # Validation
    test = 14
    x_train, y_train = seq(scaled[:-test])
    x_train = x_train.reshape(x_train.shape[0], lookback, 1)

    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(lookback, 1)),
        LSTM(50),
        Dense(25),
        Dense(1),
    ])

    model.compile(optimizer='adam', loss='mse')
    model.fit(x_train, y_train, epochs=1, batch_size=32, verbose=0)

    # Validate
    val_seq = scaled[-(lookback + test):]
    batch = val_seq[:lookback].reshape(1, lookback, 1)
    preds = []

    for _ in range(test):
        p = model.predict(batch, verbose=0)[0]
        preds.append(p)
        batch = np.append(batch[:, 1:, :], [[p]], axis=1)

    preds = scaler.inverse_transform(preds).flatten()
    actual = scaler.inverse_transform(scaled[-test:]).flatten()
    metrics = calculate_metrics(actual, preds)

    # Future
    x_all, y_all = seq(scaled)
    x_all = x_all.reshape(x_all.shape[0], lookback, 1)

    model.fit(x_all, y_all, epochs=1, batch_size=32, verbose=0)

    future = []
    batch = scaled[-lookback:].reshape(1, lookback, 1)

    for _ in range(days_forecast):
        p = model.predict(batch, verbose=0)[0]
        future.append(p)
        batch = np.append(batch[:, 1:, :], [[p]], axis=1)

    return scaler.inverse_transform(future).flatten().tolist(), metrics

In [5]:
symbols = {
    "TCS": "TCS.NS",
    "Infosys": "INFY.NS",
    "ITC": "ITC.NS",
    "YesBank": "YESBANK.NS",
    "HDFCBank": "HDFCBANK.NS",
    "Motherson": "MOTHERSON.NS",
    "PNB": "PNB.NS",
    "ICICIGI": "ICICIGI.NS",
    "Naukri": "NAUKRI.NS",
    "TVSMotor": "TVSMOTOR.NS",
    "SRF": "SRF.NS",
    "CanaraBank": "CANBK.NS",
    "Marico": "MARICO.NS",
    "HDFCLife": "HDFCLIFE.NS",
    "TorrentPharm": "TORNTPHARM.NS",
    "INDHotel": "INDHOTEL.NS",
    "JindalSteel": "JINDALSTEL.NS",
    "ShriramFin": "SHRIRAMFIN.NS",
    "ZydusLife": "ZYDUSLIFE.NS",
    "ICICIPruLI": "ICICIPRULI.NS",
    "PFC": "PFC.NS",
    "RECLtd": "RECLTD.NS",
    "AdaniEnsol": "ADANIENSOL.NS",
    "BoschLtd": "BOSCHLTD.NS",
    "Colpal": "COLPAL.NS",
    "NAMIndia": "NAM-INDIA.NS",
    "MuthootFin": "MUTHOOTFIN.NS",
    "PIIndustries": "PIIND.NS",
    "PageInd": "PAGEIND.NS"
}

start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

processed_data = {}

print(f"Downloading data for {len(symbols)} stocks...")

for name, symbol in symbols.items():
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False)

        if df.empty:
            print(f"❌ {name} returned empty data.")
            continue

        df = df.reset_index()

        # --- FIX MULTIINDEX COLUMNS ---
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join([str(c) for c in col if c]) for col in df.columns]
        else:
            df.columns = df.columns.astype(str)

        # Normalize columns to simple names
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]

        # Identify and rename date column safely
        date_col = [c for c in df.columns if "date" in c][0]
        df.rename(columns={date_col: "date"}, inplace=True)

        # Identify and rename close column safely
        close_candidates = [c for c in df.columns if "close" in c]
        if len(close_candidates) == 0:
            raise ValueError("Close column missing after flattening")
        df.rename(columns={close_candidates[0]: "close"}, inplace=True)

        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
        df["close"] = pd.to_numeric(df["close"], errors='coerce')
        df.dropna(subset=["close"], inplace=True)

        # Indicators
        df["MA50"] = df["close"].rolling(50).mean()
        df["MA200"] = df["close"].rolling(200).mean()

        # RSI
        delta = df["close"].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df["RSI"] = 100 - (100 / (1 + rs))

        df = df[["date", "close", "MA50", "MA200", "RSI"]].dropna()

        processed_data[name] = df
        print(f"✔ {name} Loaded ({len(df)} rows)")

    except Exception as e:
        print(f"❌ {name} Error: {e}")

✔ TCS Loaded (1273 rows)
✔ Infosys Loaded (1273 rows)
✔ ITC Loaded (1273 rows)
✔ YesBank Loaded (1273 rows)
✔ HDFCBank Loaded (1273 rows)
✔ Motherson Loaded (1273 rows)
✔ PNB Loaded (1273 rows)
✔ ICICIGI Loaded (1273 rows)
✔ Naukri Loaded (1273 rows)
✔ TVSMotor Loaded (1273 rows)
✔ SRF Loaded (1272 rows)
✔ CanaraBank Loaded (1273 rows)
✔ Marico Loaded (1273 rows)
✔ HDFCLife Loaded (1273 rows)
✔ TorrentPharm Loaded (1273 rows)
✔ INDHotel Loaded (1273 rows)
✔ JindalSteel Loaded (1273 rows)
✔ ShriramFin Loaded (1273 rows)
✔ ZydusLife Loaded (1273 rows)
✔ ICICIPruLI Loaded (1273 rows)
✔ PFC Loaded (1273 rows)
✔ RECLtd Loaded (1273 rows)
✔ AdaniEnsol Loaded (368 rows)
✔ BoschLtd Loaded (1272 rows)
✔ Colpal Loaded (1273 rows)
✔ NAMIndia Loaded (1273 rows)
✔ MuthootFin Loaded (1273 rows)
✔ PIIndustries Loaded (1273 rows)
✔ PageInd Loaded (1272 rows)


In [6]:
all_results = []
DAYS_FORECAST = 7

print("\nRunning all models...\n")

for company, df in processed_data.items():
    print(f"\n🔵 Company: {company}")

    models = [
        ("Linear Regression", predict_linear_regression),
        ("Random Forest", lambda x, y: predict_tree_model(x, 'rf', y)),
        ("XGBoost", lambda x, y: predict_tree_model(x, 'xgb', y)),
        ("ARIMA", predict_arima),
        ("Prophet", predict_prophet),
        ("LSTM", predict_lstm),
    ]

    for model_name, func in models:
        print(f"  → {model_name}")
        preds, mets = func(df, DAYS_FORECAST)
        if preds:
            all_results.append({"Company": company, "Model": model_name, **mets})


Running all models...


🔵 Company: TCS
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: Infosys
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: ITC
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: YesBank
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: HDFCBank
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: Motherson
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: PNB
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: ICICIGI
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: Naukri
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: TVSMotor
  → Linear Regression
  → Random Fores

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.


  → Prophet
  → LSTM

🔵 Company: BoschLtd
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: Colpal
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: NAMIndia
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: MuthootFin
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: PIIndustries
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM

🔵 Company: PageInd
  → Linear Regression
  → Random Forest
  → XGBoost
  → ARIMA
  → Prophet
  → LSTM


In [7]:
results_df = pd.DataFrame(all_results)

if results_df.empty:
    print("No results generated.")
else:
    print("🏆 Model Performance Summary")
    display(results_df.sort_values("Accuracy", ascending=False))

    print("\n🥇 Best Model per Company")
    best = results_df.loc[results_df.groupby("Company")["Accuracy"].idxmax()]
    display(best)

    results_df.to_csv("advanced_model_results.csv", index=False)
    print("\nSaved to advanced_model_results.csv")

🏆 Model Performance Summary


,Company,Model,RMSE,Accuracy
108,ZydusLife,Linear Regression,6.01,99.47
111,ZydusLife,ARIMA,7.38,99.36
78,HDFCLife,Linear Regression,8.44,99.31
15,ITC,ARIMA,3.19,99.30
81,HDFCLife,ARIMA,8.53,99.29
...,...,...,...,...
160,MuthootFin,Prophet,483.54,87.15
107,ShriramFin,LSTM,130.11,84.78
172,PageInd,Prophet,6234.05,83.77
106,ShriramFin,Prophet,163.54,80.67



🥇 Best Model per Company


,Company,Model,RMSE,Accuracy
136,AdaniEnsol,Prophet,18.02,98.55
141,BoschLtd,ARIMA,622.63,98.56
67,CanaraBank,Random Forest,2.58,98.55
145,Colpal,Random Forest,27.48,99.17
24,HDFCBank,Linear Regression,8.48,99.28
78,HDFCLife,Linear Regression,8.44,99.31
43,ICICIGI,Random Forest,30.04,98.69
116,ICICIPruLI,XGBoost,9.41,98.75
91,INDHotel,Random Forest,8.79,99.00
15,ITC,ARIMA,3.19,99.30



Saved to advanced_model_results.csv
